# Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
orders = pd.read_csv("../data/raw/orders.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
order_item_refunds = pd.read_csv("../data/raw/order_item_refunds.csv")
website_sessions = pd.read_csv("../data/raw/website_sessions.csv")
website_pageviews = pd.read_csv("../data/raw/website_pageviews.csv")
products = pd.read_csv("../data/raw/products.csv")

# Exploratory Data Analysis (EDA)

## 3.1 Business KPIs

**Objective**

Calculate high-level business metrics to understand the company's overall performance before drilling down into detailed analyses.

In [3]:
total_orders = orders.shape[0]

gross_revenue = orders["price_usd"].sum()

total_cogs = orders["cogs_usd"].sum()

total_refunds = order_item_refunds["refund_amount_usd"].sum()

net_revenue = gross_revenue - total_refunds

gross_profit = net_revenue - total_cogs

profit_margin = (gross_profit / net_revenue) * 100

average_order_value = gross_revenue / total_orders

In [4]:
print(f"Total Orders: {total_orders:,}")
print(f"Gross Revenue: ${gross_revenue:,.2f}")
print(f"Total Refunds: ${total_refunds:,.2f}")
print(f"Net Revenue: ${net_revenue:,.2f}")
print(f"Total COGS: ${total_cogs:,.2f}")
print(f"Gross Profit: ${gross_profit:,.2f}")
print(f"Profit Margin: {profit_margin:.2f}%")
print(f"Average Order Value: ${average_order_value:.2f}")

Total Orders: 32,313
Gross Revenue: $1,938,509.75
Total Refunds: $85,338.69
Net Revenue: $1,853,171.06
Total COGS: $722,370.25
Gross Profit: $1,130,800.81
Profit Margin: 61.02%
Average Order Value: $59.99


## 3.2 Product Performance Analysis

In [5]:
# merging tables with product names and prices and cogs

product_sales = order_items.merge(
    products,
    on="product_id",
    how="left"
)
product_sales.head()    

,order_item_id,created_at_x,order_id,product_id,is_primary_item,price_usd,cogs_usd,created_at_y,product_name
0,1,2012-03-19 10:42:46,1,1,1,49.99,19.49,2012-03-19 08:00:00,The Original Mr. Fuzzy
1,2,2012-03-19 19:27:37,2,1,1,49.99,19.49,2012-03-19 08:00:00,The Original Mr. Fuzzy
2,3,2012-03-20 06:44:45,3,1,1,49.99,19.49,2012-03-19 08:00:00,The Original Mr. Fuzzy
3,4,2012-03-20 09:41:45,4,1,1,49.99,19.49,2012-03-19 08:00:00,The Original Mr. Fuzzy
4,5,2012-03-20 11:28:15,5,1,1,49.99,19.49,2012-03-19 08:00:00,The Original Mr. Fuzzy


In [6]:
# Calculating revenue

product_revenue = (
    product_sales
        .groupby("product_name")
        .agg(
            Revenue=("price_usd", "sum")
        )
        .sort_values(
            by="Revenue",
            ascending=False
        )
)

product_revenue

,Revenue
product_name,
The Original Mr. Fuzzy,1211057.74
The Forever Love Bear,347702.04
The Birthday Sugar Panda,229260.15
The Hudson River Mini bear,150489.82


In [7]:
# Calculating most generated profit

product_profit = (
    product_sales
        .groupby("product_name")
        .agg(
            Revenue=("price_usd", "sum"),
            COGS=("cogs_usd", "sum")
        )
)

product_profit["Profit"] = (
    product_profit["Revenue"]
    - product_profit["COGS"]
)

product_profit = product_profit.sort_values(
    by="Profit",
    ascending=False
)

product_profit

,Revenue,COGS,Profit
product_name,,,
The Original Mr. Fuzzy,1211057.74,472164.74,738893.0
The Forever Love Bear,347702.04,130352.04,217350.0
The Birthday Sugar Panda,229260.15,72232.65,157027.5
The Hudson River Mini bear,150489.82,47620.82,102869.0


In [8]:
# Calculating products are sold the most

product_units = (
    product_sales
        .groupby("product_name")
        .agg(
            units_sold=("order_item_id", "count")
        )
        .sort_values(by="units_sold", ascending=False)
)

product_units

,units_sold
product_name,
The Original Mr. Fuzzy,24226
The Forever Love Bear,5796
The Hudson River Mini bear,5018
The Birthday Sugar Panda,4985


In [9]:
# refund analysis amount

refund_analysis = order_item_refunds.merge(
    product_sales,
    on="order_item_id",
    how="left"
)

print(list(refund_analysis.columns))

product_refunds = (
    refund_analysis
        .groupby("product_name")
        .agg(
            Refund_Amount=("refund_amount_usd", "sum")
        )
        .sort_values(by="Refund_Amount", ascending=False)
)

product_refunds

['order_item_refund_id', 'created_at', 'order_item_id', 'order_id_x', 'refund_amount_usd', 'created_at_x', 'order_id_y', 'product_id', 'is_primary_item', 'price_usd', 'cogs_usd', 'created_at_y', 'product_name']


,Refund_Amount
product_name,
The Original Mr. Fuzzy,61837.63
The Birthday Sugar Panda,13842.99
The Forever Love Bear,7738.71
The Hudson River Mini bear,1919.36


In [10]:
# refund analysis count 

refund_count = (
    refund_analysis
        .groupby("product_name")
        .agg(
            refund_count=("order_item_refund_id", "count")
        )
        .sort_values(by="refund_count", ascending=False)
)

refund_count

,refund_count
product_name,
The Original Mr. Fuzzy,1237
The Birthday Sugar Panda,301
The Forever Love Bear,129
The Hudson River Mini bear,64


In [11]:
# refund analysis refund rate

refund_rate = product_units.merge(
    refund_count,
    on="product_name",
    how="left"
)

refund_rate["refund_rate"] = (refund_count["refund_count"] / product_units["units_sold"])

refund_rate.head(5)

,units_sold,refund_count,refund_rate
product_name,,,
The Original Mr. Fuzzy,24226,1237,0.051061
The Forever Love Bear,5796,129,0.022257
The Hudson River Mini bear,5018,64,0.012754
The Birthday Sugar Panda,4985,301,0.060381


In [12]:
# revenue contribution

total_revenue = product_revenue["Revenue"].sum()
product_revenue["Revenue_Contribution"] = (product_revenue["Revenue"] / total_revenue)
product_revenue

,Revenue,Revenue_Contribution
product_name,,
The Original Mr. Fuzzy,1211057.74,0.624736
The Forever Love Bear,347702.04,0.179366
The Birthday Sugar Panda,229260.15,0.118266
The Hudson River Mini bear,150489.82,0.077632


In [13]:
# Profit Contribution

total_profit = product_profit["Profit"].sum()

product_profit["Profit_Contribution"] = (
    product_profit["Profit"] / total_profit
) * 100

product_profit

,Revenue,COGS,Profit,Profit_Contribution
product_name,,,,
The Original Mr. Fuzzy,1211057.74,472164.74,738893.0,60.757257
The Forever Love Bear,347702.04,130352.04,217350.0,17.872127
The Birthday Sugar Panda,229260.15,72232.65,157027.5,12.911964
The Hudson River Mini bear,150489.82,47620.82,102869.0,8.458651


In [14]:
# best selling product

best_selling_product = product_units.head(1)

best_selling_product

,units_sold
product_name,
The Original Mr. Fuzzy,24226


In [15]:
# least selling project

least_selling_product = product_units.tail(1)

least_selling_product

,units_sold
product_name,
The Birthday Sugar Panda,4985


In [16]:
# product ranking

product_revenue["Rank"] = (
    product_revenue["Revenue"]
        .rank(method="dense", ascending=False)
        .astype(int)
)

product_revenue

,Revenue,Revenue_Contribution,Rank
product_name,,,
The Original Mr. Fuzzy,1211057.74,0.624736,1
The Forever Love Bear,347702.04,0.179366,2
The Birthday Sugar Panda,229260.15,0.118266,3
The Hudson River Mini bear,150489.82,0.077632,4


In [17]:
product_summary = (
    product_revenue
        .merge(
            product_profit[["Profit"]],
            on="product_name",
            how="left"
        )
        .merge(
            product_units,
            on="product_name",
            how="left"
        )
        .merge(
            product_refunds,
            on="product_name",
            how="left"
        )
        .merge(
            refund_count,
            on="product_name",
            how="left"
        )
        .merge(
            refund_rate[["refund_rate"]],
            on="product_name",
            how="left"
        )
)

product_summary[["Refund_Amount", "refund_count", "refund_rate"]] = (
    product_summary[["Refund_Amount", "refund_count", "refund_rate"]].fillna(0)
)

product_summary["Rank"] = (
    product_summary["Revenue"]
        .rank(method="dense", ascending=False)
        .astype(int)
)

product_summary = product_summary.sort_values(
    by="Revenue",
    ascending=False
)

product_summary

,Revenue,Revenue_Contribution,Rank,Profit,units_sold,Refund_Amount,refund_count,refund_rate
product_name,,,,,,,,
The Original Mr. Fuzzy,1211057.74,0.624736,1,738893.0,24226,61837.63,1237,0.051061
The Forever Love Bear,347702.04,0.179366,2,217350.0,5796,7738.71,129,0.022257
The Birthday Sugar Panda,229260.15,0.118266,3,157027.5,4985,13842.99,301,0.060381
The Hudson River Mini bear,150489.82,0.077632,4,102869.0,5018,1919.36,64,0.012754


## 3.3 Customer Analysis

In [18]:
# Total customers

total_customers = orders["user_id"].nunique()

print(f"Total Customers: {total_customers:,}")

Total Customers: 31,696


In [19]:
# New vs Returning Customers

customer_type = (
    website_sessions["is_repeat_session"]
        .value_counts()
        .rename(index={
            0: "New Customers",
            1: "Returning Customers"
        })
)

customer_type

is_repeat_session
New Customers          394318
Returning Customers     78553
Name: count, dtype: int64

In [20]:
# Repeat Purchase Rate
#
# NOTE: This must be computed at CUSTOMER grain, not SESSION grain.
# The previous version divided a session count
# (website_sessions["is_repeat_session"] == 1 -> 78,553 repeat *sessions*)
# by the number of unique *customers* (31,696), which mixed grains and
# produced an impossible 247.83%.
#
# Correct definition: share of customers who placed more than one order.
orders_per_user = (
    orders
        .groupby("user_id")
        .agg(Orders=("order_id", "count"))
)

repeat_customers = (orders_per_user["Orders"] > 1).sum()

repeat_purchase_rate = (repeat_customers / total_customers) * 100

print(f"Repeat Customers: {repeat_customers:,}")
print(f"Repeat Purchase Rate: {repeat_purchase_rate:.2f}%")

Repeat Customers: 591
Repeat Purchase Rate: 1.86%


In [21]:
# Orders per Customer

orders_per_customer = (
    orders
        .groupby("user_id", as_index=False)
        .agg(
            Total_Orders=("order_id", "count")
        )
        .sort_values(by="Total_Orders", ascending=False)
)

orders_per_customer

,user_id,Total_Orders
21809,289300,3
4753,83383,3
10652,158790,3
18129,248399,3
10607,158032,3
...,...,...
10634,158521,1
10633,158490,1
10632,158484,1
10631,158454,1


In [22]:
# Average Revenue per Customer

customer_revenue = (
    orders
        .groupby("user_id", as_index=False)
        .agg(
            Revenue=("price_usd", "sum")
        )
)

average_revenue_per_customer = customer_revenue["Revenue"].mean()

print(f"Average Revenue per Customer: ${average_revenue_per_customer:.2f}")

Average Revenue per Customer: $61.16


In [23]:
# Highest Spending Customers

highest_spending_customers = (
    customer_revenue
        .sort_values(
            by="Revenue",
            ascending=False
        )
)

highest_spending_customers.head(10)

,user_id,Revenue
26808,341972,251.94
25167,324814,245.95
11722,172266,237.95
20506,275098,235.95
21062,281298,219.96
9314,142410,219.96
24501,317773,215.96
26264,336150,209.95
10652,158790,205.96
23498,307597,205.96


In [24]:
# Customer Segmentation

customer_revenue["Customer_Segment"] = pd.qcut(
    customer_revenue["Revenue"],
    q=3,
    labels=["Low Value", "Medium Value", "High Value"]
)

customer_revenue

,user_id,Revenue,Customer_Segment
0,13,49.99,Low Value
1,20,49.99,Low Value
2,59,49.99,Low Value
3,104,49.99,Low Value
4,147,49.99,Low Value
...,...,...,...
31691,394231,49.99,Low Value
31692,394255,49.99,Low Value
31693,394257,29.99,Low Value
31694,394268,89.98,High Value


In [25]:
# Customer Lifetime Value (Basic)

customer_lifetime_value = (
    gross_revenue / total_customers
)

print(f"Customer Lifetime Value (Basic): ${customer_lifetime_value:.2f}")

Customer Lifetime Value (Basic): $61.16


In [26]:
# Customer Summary

customer_summary = (
    customer_revenue
        .merge(
            orders_per_customer,
            on="user_id"
        )
)

customer_summary["Customer_Lifetime_Value"] = (
    customer_summary["Revenue"]
)

customer_summary = customer_summary.sort_values(
    by="Revenue",
    ascending=False
)

customer_summary

,user_id,Revenue,Customer_Segment,Total_Orders,Customer_Lifetime_Value
26808,341972,251.94,High Value,3,251.94
25167,324814,245.95,High Value,3,245.95
11722,172266,237.95,High Value,3,237.95
20506,275098,235.95,High Value,3,235.95
21062,281298,219.96,High Value,2,219.96
...,...,...,...,...,...
25469,327952,29.99,Low Value,1,29.99
25468,327917,29.99,Low Value,1,29.99
29517,369596,29.99,Low Value,1,29.99
22775,299952,29.99,Low Value,1,29.99


In [27]:
customer_analysis = customer_summary.copy()

In [28]:
customer_analysis.to_csv(
    "../data/processed/customers/customer_summary.csv",
    index=False
)

customer_analysis.head()

,user_id,Revenue,Customer_Segment,Total_Orders,Customer_Lifetime_Value
26808,341972,251.94,High Value,3,251.94
25167,324814,245.95,High Value,3,245.95
11722,172266,237.95,High Value,3,237.95
20506,275098,235.95,High Value,3,235.95
21062,281298,219.96,High Value,2,219.96


## 3.4 Sales Analysis

### Objective

Analyze sales trends over time to identify seasonality, growth, and customer purchasing patterns.

In [29]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32313 entries, 0 to 32312
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            32313 non-null  int64  
 1   created_at          32313 non-null  object 
 2   website_session_id  32313 non-null  int64  
 3   user_id             32313 non-null  int64  
 4   primary_product_id  32313 non-null  int64  
 5   items_purchased     32313 non-null  int64  
 6   price_usd           32313 non-null  float64
 7   cogs_usd            32313 non-null  float64
dtypes: float64(2), int64(5), object(1)
memory usage: 2.0+ MB


In [30]:
# Convert created_at to datetime

orders["created_at"] = pd.to_datetime(
    orders["created_at"]
)

orders.dtypes

order_id                       int64
created_at            datetime64[ns]
website_session_id             int64
user_id                        int64
primary_product_id             int64
items_purchased                int64
price_usd                    float64
cogs_usd                     float64
dtype: object

In [31]:
orders["created_at"].dt.month

# Extract Date Features

orders["Year"] = orders["created_at"].dt.year
orders["Month"] = orders["created_at"].dt.month_name()
orders["Month_Number"] = orders["created_at"].dt.month
orders["Week"] = orders["created_at"].dt.isocalendar().week
orders["Day"] = orders["created_at"].dt.day_name()
orders["Hour"] = orders["created_at"].dt.hour
orders["Date"] = orders["created_at"].dt.date

orders.head()

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd,Year,Month,Month_Number,Week,Day,Hour,Date
0,1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49,2012,March,3,12,Monday,10,2012-03-19
1,2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49,2012,March,3,12,Monday,19,2012-03-19
2,3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49,2012,March,3,12,Tuesday,6,2012-03-20
3,4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49,2012,March,3,12,Tuesday,9,2012-03-20
4,5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49,2012,March,3,12,Tuesday,11,2012-03-20


In [32]:
# 3.4.1 Orders by Year

yearly_orders = (
    orders
    .groupby("Year", as_index=False)
    .agg(
        Orders=("order_id", "count")
    )
)

yearly_orders.to_csv(
    "../data/processed/sales/yearly_orders.csv",
    index=False
)

yearly_orders

,Year,Orders
0,2012,2586
1,2013,7447
2,2014,16860
3,2015,5420


In [33]:
# 3.4.2 Revenue by Year

yearly_revenue = (
    orders
    .groupby("Year", as_index=False)
    .agg(
        Revenue=("price_usd", "sum")
    )
)

yearly_revenue.to_csv(
    "../data/processed/sales/yearly_revenue.csv",
    index=False
)

yearly_revenue

,Year,Revenue
0,2012,129274.14
1,2013,393247.87
2,2014,1075612.19
3,2015,340375.55


In [34]:
# 3.4.3 Orders by Month

monthly_orders = (
    orders
    .groupby(["Year", "Month_Number", "Month"], as_index=False)
    .agg(
        Orders=("order_id", "count")
    )
    .sort_values(["Year", "Month_Number"])
)

monthly_orders.to_csv(
    "../data/processed/sales/monthly_orders.csv",
    index=False
)

monthly_orders.head()

,Year,Month_Number,Month,Orders
0,2012,3,March,60
1,2012,4,April,99
2,2012,5,May,108
3,2012,6,June,140
4,2012,7,July,169


In [35]:
# 3.4.4 Revenue by Month

monthly_revenue = (
    orders
    .groupby(["Year", "Month_Number", "Month"], as_index=False)
    .agg(
        Revenue=("price_usd", "sum")
    )
    .sort_values(["Year", "Month_Number"])
)

monthly_revenue.to_csv(
    "../data/processed/sales/monthly_revenue.csv",
    index=False
)

monthly_revenue.head()

,Year,Month_Number,Month,Revenue
0,2012,3,March,2999.40
1,2012,4,April,4949.01
2,2012,5,May,5398.92
3,2012,6,June,6998.60
4,2012,7,July,8448.31


In [36]:
# 3.4.5 Orders by Week

weekly_orders = (
    orders
    .groupby(pd.Grouper(key="created_at", freq="W"))
    .agg(
        Orders=("order_id", "count")
    )
    .reset_index()
)

weekly_orders.to_csv(
    "../data/processed/sales/weekly_orders.csv",
    index=False
)

weekly_orders.head()

,created_at,Orders
0,2012-03-25,29
1,2012-04-01,35
2,2012-04-08,30
3,2012-04-15,27
4,2012-04-22,21


In [37]:
# 3.4.6 Revenue by Week

weekly_revenue = (
    orders
    .groupby(pd.Grouper(key="created_at", freq="W"))
    .agg(
        Revenue=("price_usd", "sum")
    )
    .reset_index()
)

weekly_revenue.to_csv(
    "../data/processed/sales/weekly_revenue.csv",
    index=False
)

weekly_revenue.head()

,created_at,Revenue
0,2012-03-25,1449.71
1,2012-04-01,1749.65
2,2012-04-08,1499.70
3,2012-04-15,1349.73
4,2012-04-22,1049.79


In [38]:
# 3.4.6 Orders by Day

daily_orders = (
    orders
    .groupby(orders["created_at"].dt.date)
    .agg(
        Orders=("order_id", "count")
    )
    .reset_index(names="Date")
)

daily_orders.to_csv(
    "../data/processed/sales/daily_orders.csv",
    index=False
)

In [39]:
# 3.4.7 Revenue by Day

daily_revenue = (
    orders
    .groupby(orders["created_at"].dt.date)
    .agg(
        Revenue=("price_usd", "sum")
    )
    .reset_index(names="Date")
)

daily_revenue.to_csv(
    "../data/processed/sales/daily_revenue.csv",
    index=False
)

In [40]:
# 3.4.9 Revenue by Hour

hourly_revenue = (
    orders
    .groupby("Hour")
    .agg(
        Revenue=("price_usd", "sum")
    )
    .reset_index()
)

hourly_revenue.to_csv(
    "../data/processed/sales/hourly_revenue.csv",
    index=False
)

hourly_revenue.head()

,Hour,Revenue
0,0,49157.84
1,1,43287.01
2,2,35810.62
3,3,32299.37
4,4,33317.11


In [41]:
# 3.4.10 Sales Summary

sales_summary = monthly_revenue.merge(
    monthly_orders,
    on=["Year", "Month_Number", "Month"],
    how="left"
)

sales_summary["Date"] = pd.to_datetime(
    dict(
        year=sales_summary["Year"],
        month=sales_summary["Month_Number"],
        day=1
    )
)

sales_summary.to_csv(
    "../data/processed/sales/sales_summary.csv",
    index=False
)
sales_summary

# print(monthly_revenue.columns)    
# print(monthly_orders.columns)

,Year,Month_Number,Month,Revenue,Orders,Date
0,2012,3,March,2999.40,60,2012-03-01
1,2012,4,April,4949.01,99,2012-04-01
2,2012,5,May,5398.92,108,2012-05-01
3,2012,6,June,6998.60,140,2012-06-01
4,2012,7,July,8448.31,169,2012-07-01
5,2012,8,August,11397.72,228,2012-08-01
6,2012,9,September,14347.13,287,2012-09-01
7,2012,10,October,18546.29,371,2012-10-01
8,2012,11,November,30893.82,618,2012-11-01
9,2012,12,December,25294.94,506,2012-12-01


In [42]:
# 3.4.11 Data Cleaning for Marketing & Website Analysis

website_sessions["utm_source"] = (
    website_sessions["utm_source"]
    .fillna("Direct")
)

website_sessions["utm_campaign"] = (
    website_sessions["utm_campaign"]
    .fillna("No Campaign")
)

website_sessions["device_type"] = (
    website_sessions["device_type"]
    .fillna("Unknown Device")
)

website_sessions["http_referer"] = (
    website_sessions["http_referer"]
    .fillna("Direct")
)

website_pageviews["pageview_url"] = (
    website_pageviews["pageview_url"]
    .fillna("Unknown Page")
)

website_pageviews.isna().sum()

website_pageview_id    0
created_at             0
website_session_id     0
pageview_url           0
dtype: int64

## 3.5 Marketing Analysis

In [43]:
# Merge sessions with orders

marketing_df = website_sessions.merge(
    orders[["website_session_id", "order_id", "price_usd"]],
    on="website_session_id",
    how="left"
)

marketing_df.head()

,website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer,order_id,price_usd
0,1,2012-03-19 08:04:16,1,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com,NaN,NaN
1,2,2012-03-19 08:16:49,2,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,NaN,NaN
2,3,2012-03-19 08:26:55,3,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,NaN,NaN
3,4,2012-03-19 08:37:33,4,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,NaN,NaN
4,5,2012-03-19 09:00:55,5,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com,NaN,NaN


In [44]:
# 3.5.1 Sessions by Source

sessions_by_source = (
    marketing_df
    .groupby("utm_source", dropna=False)
    .agg(
        Sessions=("website_session_id", "count")
    )
    .reset_index()
    .sort_values("Sessions", ascending=False)
)

sessions_by_source["utm_source"] = (
    sessions_by_source["utm_source"]
    .fillna("Direct")
)

sessions_by_source.to_csv(
    "../data/processed/marketing/sessions_by_source.csv",
    index=False
)

sessions_by_source

,utm_source,Sessions
2,gsearch,316035
0,Direct,83328
1,bsearch,62823
3,socialbook,10685


In [45]:
# 3.5.2 Sessions by Campaign

sessions_by_campaign = (
    marketing_df
    .groupby("utm_campaign", dropna=False)
    .agg(
        Sessions=("website_session_id", "count")
    )
    .reset_index()
    .sort_values("Sessions", ascending=False)
)

sessions_by_campaign["utm_campaign"] = (
    sessions_by_campaign["utm_campaign"]
    .fillna("Direct")
)

sessions_by_campaign.to_csv(
    "../data/processed/marketing/sessions_by_campaign.csv",
    index=False
)

sessions_by_campaign

,utm_campaign,Sessions
3,nonbrand,337615
0,No Campaign,83328
1,brand,41243
2,desktop_targeted,5590
4,pilot,5095


In [46]:
# 3.5.3 Sessions by Device

sessions_by_device = (
    marketing_df
    .groupby("device_type")
    .agg(
        Sessions=("website_session_id", "count")
    )
    .reset_index()
    .sort_values("Sessions", ascending=False)
)

sessions_by_device.to_csv(
    "../data/processed/marketing/sessions_by_device.csv",
    index=False
)

sessions_by_device

,device_type,Sessions
0,desktop,327027
1,mobile,145844


In [47]:
# Marketing Dataset

marketing_df = website_sessions.merge(
    orders,
    on="website_session_id",
    how="left"
)

marketing_df.head()

,website_session_id,created_at_x,user_id_x,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer,order_id,...,items_purchased,price_usd,cogs_usd,Year,Month,Month_Number,Week,Day,Hour,Date
0,1,2012-03-19 08:04:16,1,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
1,2,2012-03-19 08:16:49,2,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
2,3,2012-03-19 08:26:55,3,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
3,4,2012-03-19 08:37:33,4,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
4,5,2012-03-19 09:00:55,5,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN


In [48]:
# 3.5.4 Revenue by Source

revenue_by_source = (
    marketing_df
    .groupby("utm_source", dropna=False)
    .agg(
        Revenue=("price_usd", "sum")
    )
    .reset_index()
    .sort_values("Revenue", ascending=False)
)

revenue_by_source["utm_source"] = (
    revenue_by_source["utm_source"]
    .fillna("Direct")
)

revenue_by_source.to_csv(
    "../data/processed/marketing/revenue_by_source.csv",
    index=False
)

revenue_by_source

,utm_source,Revenue
2,gsearch,1276144.89
0,Direct,371433.03
1,bsearch,268672.50
3,socialbook,22259.33


In [49]:
# 3.5.5 Revenue by Campaign

revenue_by_campaign = (
    marketing_df
    .groupby("utm_campaign", dropna=False)
    .agg(
        Revenue=("price_usd", "sum")
    )
    .reset_index()
    .sort_values("Revenue", ascending=False)
)

revenue_by_campaign["utm_campaign"] = (
    revenue_by_campaign["utm_campaign"]
    .fillna("Direct")
)

revenue_by_campaign.to_csv(
    "../data/processed/marketing/revenue_by_campaign.csv",
    index=False
)

revenue_by_campaign

,utm_campaign,Revenue
3,nonbrand,1349977.69
0,No Campaign,371433.03
1,brand,194839.70
2,desktop_targeted,18516.10
4,pilot,3743.23


In [50]:
# 3.5.6 Conversion Rate

conversion_rate = (
    marketing_df
    .groupby("utm_source", dropna=False)
    .agg(
        Sessions=("website_session_id", "count"),
        Orders=("order_id", "count")
    )
    .reset_index()
)

conversion_rate["Conversion Rate"] = (
    conversion_rate["Orders"]
    / conversion_rate["Sessions"]
)

conversion_rate["utm_source"] = (
    conversion_rate["utm_source"]
    .fillna("Direct")
)

conversion_rate = conversion_rate.sort_values(
    "Conversion Rate",
    ascending=False
)

conversion_rate.to_csv(
    "../data/processed/marketing/conversion_rate.csv",
    index=False
)

conversion_rate

,utm_source,Sessions,Orders,Conversion Rate
0,Direct,83328,6118,0.073421
1,bsearch,62823,4519,0.071932
2,gsearch,316035,21333,0.067502
3,socialbook,10685,343,0.032101


In [51]:
# 3.5.7 Device Performance

device_revenue = (
    marketing_df
    .groupby("device_type")
    .agg(
        Revenue=("price_usd", "sum")
    )
    .reset_index()
    .sort_values("Revenue", ascending=False)
)

device_revenue.to_csv(
    "../data/processed/marketing/device_revenue.csv",
    index=False
)

device_revenue

,device_type,Revenue
0,desktop,1665757.84
1,mobile,272751.91


In [52]:
# # 3.5.8 Device Conversion rate

device_conversion_rate = (
    marketing_df
    .groupby("device_type")
    .agg(
        Sessions=("website_session_id", "count"),
        Orders=("order_id", "count")
    )
    .reset_index()
)

device_conversion_rate["Conversion Rate"] = (
    device_conversion_rate["Orders"]
    / device_conversion_rate["Sessions"]
)

device_conversion_rate.to_csv(
    "../data/processed/marketing/device_conversion_rate.csv",
    index=False
)

device_conversion_rate

,device_type,Sessions,Orders,Conversion Rate
0,desktop,327027,27805,0.085024
1,mobile,145844,4508,0.030910


In [53]:
# Marketing Summary

marketing_summary = (
    sessions_by_source
    .merge(
        revenue_by_source,
        on="utm_source",
        how="left"
    )
    .merge(
        conversion_rate[["utm_source","Conversion Rate"]],
        on="utm_source",
        how="left"
    )
)

marketing_summary

,utm_source,Sessions,Revenue,Conversion Rate
0,gsearch,316035,1276144.89,0.067502
1,Direct,83328,371433.03,0.073421
2,bsearch,62823,268672.50,0.071932
3,socialbook,10685,22259.33,0.032101


## 3.6 Website Funnel Analysis

In [54]:
# First Pageview of Each Session

first_pageviews = (
    website_pageviews
        .groupby("website_session_id")["website_pageview_id"]
        .idxmin()
)

first_pageviews.head()

website_session_id
1    0
2    1
3    2
4    3
5    4
Name: website_pageview_id, dtype: int64

In [55]:
# 3.6.1 Landing Pages

landing_pages = (
    website_pageviews
    .sort_values("created_at")
    .groupby("website_session_id")
    .first()
    .reset_index()
)

landing_pages = (
    landing_pages
    .groupby("pageview_url", as_index=False)
    .agg(
        Sessions=("website_session_id", "count")
    )
    .sort_values("Sessions", ascending=False)
)

landing_pages["pageview_url"] = (
    landing_pages["pageview_url"]
    .fillna("Unknown Page")
)

landing_pages.to_csv(
    "../data/processed/website/landing_pages.csv",
    index=False
)

landing_pages.head()

,pageview_url,Sessions
0,/home,137576
2,/lander-2,131170
3,/lander-3,79000
5,/lander-5,68166
1,/lander-1,47574


In [56]:
# 3.6.2 Most Viewed Pages

most_viewed_pages = (
    website_pageviews
    .groupby("pageview_url", as_index=False)
    .agg(
        Pageviews=("website_pageview_id", "count")
    )
    .sort_values("Pageviews", ascending=False)
)

most_viewed_pages["pageview_url"] = (
    most_viewed_pages["pageview_url"]
    .fillna("Unknown Page")
)

most_viewed_pages.to_csv(
    "../data/processed/website/most_viewed_pages.csv",
    index=False
)

most_viewed_pages.head()

,pageview_url,Pageviews
9,/products,261231
15,/the-original-mr-fuzzy,162525
3,/home,137576
5,/lander-2,131170
2,/cart,94953


In [57]:
# Last Pageview of Each Session

# 3.6.3 Exit Pages

exit_pages = (
    website_pageviews
    .sort_values("created_at")
    .groupby("website_session_id")
    .last()
    .reset_index()
)

exit_pages = (
    exit_pages
    .groupby("pageview_url", as_index=False)
    .agg(
        Exits=("website_session_id", "count")
    )
    .sort_values("Exits", ascending=False)
)

exit_pages["pageview_url"] = (
    exit_pages["pageview_url"]
    .fillna("Unknown Page")
)

exit_pages.to_csv(
    "../data/processed/website/exit_pages.csv",
    index=False
)

exit_pages.head()

,pageview_url,Exits
15,/the-original-mr-fuzzy,92568
5,/lander-2,59249
3,/home,57346
9,/products,51017
6,/lander-3,39733


In [58]:
# 3.6.4 Conversion Funnel

funnel_pages = {
    "Home": ["/home", "/lander-1", "/lander-2", "/lander-3", "/lander-4", "/lander-5"],
    "Products": ["/products"],
    "Product Page": ["/the-original-mr-fuzzy", "/the-forever-love-bear", "/the-hudson-river-mini-bear", "/the-birthday-sugar-panda"],
    "Cart": ["/cart"],
    "Shipping": ["/shipping"],
    "Billing": ["/billing", "/billing-2"],
    "Thank You": ["/thank-you-for-your-order"]
}

session_paths = (
    website_pageviews
    .sort_values(["website_session_id", "created_at"])
    .groupby("website_session_id")["pageview_url"]
    .apply(list)
)

funnel = []
current_sessions = {session_id: -1 for session_id in session_paths.index}

for stage, pages in funnel_pages.items():
    next_sessions = {}

    for session_id, last_position in current_sessions.items():
        path = session_paths.loc[session_id]

        for position, page in enumerate(path):
            if position > last_position and page in pages:
                next_sessions[session_id] = position
                break

    funnel.append([stage, len(next_sessions)])
    current_sessions = next_sessions

conversion_funnel = pd.DataFrame(
    funnel,
    columns=["Stage", "Sessions"]
)

conversion_funnel.to_csv(
    "../data/processed/website/conversion_funnel.csv",
    index=False
)

conversion_funnel

,Stage,Sessions
0,Home,472871
1,Products,261231
2,Product Page,210214
3,Cart,94953
4,Shipping,64484
5,Billing,52058
6,Thank You,32313


In [59]:
# Conversion Funnel

conversion_funnel

,Stage,Sessions
0,Home,472871
1,Products,261231
2,Product Page,210214
3,Cart,94953
4,Shipping,64484
5,Billing,52058
6,Thank You,32313


In [60]:
# 3.6.5 Drop-off Analysis

dropoff_analysis = conversion_funnel.copy()

dropoff_analysis["Drop-off (%)"] = (
    1 - dropoff_analysis["Sessions"] / dropoff_analysis["Sessions"].shift(1)
).round(4)

dropoff_analysis.loc[0, "Drop-off (%)"] = 0

dropoff_analysis.to_csv(
    "../data/processed/website/dropoff_analysis.csv",
    index=False
)

dropoff_analysis

,Stage,Sessions,Drop-off (%)
0,Home,472871,0.0000
1,Products,261231,0.4476
2,Product Page,210214,0.1953
3,Cart,94953,0.5483
4,Shipping,64484,0.3209
5,Billing,52058,0.1927
6,Thank You,32313,0.3793


In [61]:
# 3.6.6 Product Page Visits

product_pages = [
    "/the-original-mr-fuzzy",
    "/the-forever-love-bear",
    "/the-hudson-river-mini-bear",
    "/the-birthday-sugar-panda"
]

product_page_visits = (
    website_pageviews[
        website_pageviews["pageview_url"].isin(product_pages)
    ]
    .groupby("pageview_url", as_index=False)
    .agg(
        Visits=("website_session_id", "nunique")
    )
    .sort_values("Visits", ascending=False)
)

product_page_visits.to_csv(
    "../data/processed/website/product_page_visits.csv",
    index=False
)

product_page_visits

,pageview_url,Visits
3,/the-original-mr-fuzzy,162525
1,/the-forever-love-bear,26033
0,/the-birthday-sugar-panda,19046
2,/the-hudson-river-mini-bear,2610


In [62]:
# 3.6.7 Cart Analysis

cart_analysis = (
    website_pageviews[
        website_pageviews["pageview_url"] == "/cart"
    ]
    .groupby("pageview_url", as_index=False)
    .agg(
        Cart_Visits=("website_session_id", "nunique")
    )
)

cart_analysis.to_csv(
    "../data/processed/website/cart_analysis.csv",
    index=False
)

cart_analysis

,pageview_url,Cart_Visits
0,/cart,94953


In [63]:
# 3.6.8 Billing Analysis

billing_analysis = (
    website_pageviews[
        website_pageviews["pageview_url"] == "/billing"
    ]
    .groupby("pageview_url", as_index=False)
    .agg(
        Billing_Visits=("website_session_id", "nunique")
    )
)

billing_analysis.to_csv(
    "../data/processed/website/billing_analysis.csv",
    index=False
)

billing_analysis

,pageview_url,Billing_Visits
0,/billing,3617


In [64]:
# 3.6.9 Website Summary

website_summary = (
    landing_pages
    .merge(
        most_viewed_pages,
        on="pageview_url",
        how="outer"
    )
    .merge(
        exit_pages,
        on="pageview_url",
        how="outer"
    )
)

website_summary = website_summary.fillna(0)

website_summary.to_csv(
    "../data/processed/website/website_summary.csv",
    index=False
)

website_summary.head()

,pageview_url,Sessions,Pageviews,Exits
0,/billing,0.0,3617,1997
1,/billing-2,0.0,48441,17748
2,/cart,0.0,94953,30469
3,/home,137576.0,137576,57346
4,/lander-1,47574.0,47574,25330


In [65]:
# Export Processed Data
# Note: customer_summary.csv is written once, upstream in the "Customer Summary"
# section (cell that writes customer_analysis with index=False). It is
# intentionally NOT re-written here, so the file has a single, deterministic write.

product_summary.to_csv(
    "../data/processed/products/product_summary.csv",
)

marketing_summary.to_csv(
    "../data/processed/marketing/marketing_summary.csv",
    index=False
)

website_summary.to_csv(
    "../data/processed/website/website_summary.csv",
    index=False
)

In [66]:
import os
from pathlib import Path

print("Current Folder:")
print(os.getcwd())

print("\nCustomer Summary CSV:")
print(Path("../data/processed/customers/customer_summary.csv").resolve())

Current Folder:
c:\Users\KAIF\OneDrive\Desktop\E-commerce_sales_analytics\notebooks

Customer Summary CSV:
C:\Users\KAIF\OneDrive\Desktop\E-commerce_sales_analytics\data\processed\customers\customer_summary.csv


In [67]:
print(customer_summary.columns.tolist())

['user_id', 'Revenue', 'Customer_Segment', 'Total_Orders', 'Customer_Lifetime_Value']


## 3.7  Business Insights

In [68]:
website_summary

,pageview_url,Sessions,Pageviews,Exits
0,/billing,0.0,3617,1997
1,/billing-2,0.0,48441,17748
2,/cart,0.0,94953,30469
3,/home,137576.0,137576,57346
4,/lander-1,47574.0,47574,25330
5,/lander-2,131170.0,131170,59249
6,/lander-3,79000.0,79000,39733
7,/lander-4,9385.0,9385,4851
8,/lander-5,68166.0,68166,25131
9,/products,0.0,261231,51017


In [69]:
# 3.7.1 Customer Summary

customer_kpi_summary = pd.DataFrame({

    "Metric": [
        "Total Customers",
        "Total Orders",
        "Average Orders per Customer",
        "Average Revenue per Customer"
    ],

    "Value": [

        orders["user_id"].nunique(),

        orders["order_id"].nunique(),

        round(
            orders["order_id"].nunique() /
            orders["user_id"].nunique(),
            2
        ),

        round(
            orders["price_usd"].sum() /
            orders["user_id"].nunique(),
            2
        )

    ]

})

customer_kpi_summary.to_csv(
    "../data/processed/customers/customer_kpi_summary.csv",
    index=False
)

customer_kpi_summary

,Metric,Value
0,Total Customers,31696.00
1,Total Orders,32313.00
2,Average Orders per Customer,1.02
3,Average Revenue per Customer,61.16


In [70]:
# 3.7.2 Top Customers by Revenue

top_customers = (
    orders
    .groupby("user_id", as_index=False)
    .agg(
        Orders=("order_id", "count"),
        Revenue=("price_usd", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

top_customers.to_csv(
    "../data/processed/customers/top_customers.csv",
    index=False
)

top_customers.head()
pd.read_csv("../data/processed/customers/top_customers.csv").head()

,user_id,Orders,Revenue
0,341972,3,251.94
1,324814,3,245.95
2,172266,3,237.95
3,275098,3,235.95
4,281298,2,219.96


In [71]:
# 3.7.3 Customer Order Frequency

customer_order_frequency = (
    orders
    .groupby("user_id", as_index=False)
    .agg(
        Orders=("order_id", "count")
    )
)

customer_order_frequency.to_csv(
    "../data/processed/customers/customer_order_frequency.csv",
    index=False
)

customer_order_frequency.head()

,user_id,Orders
0,13,1
1,20,1
2,59,1
3,104,1
4,147,1


In [72]:
# 3.7.4 Repeat vs New Customers

customer_type = (
    customer_order_frequency
    .assign(
        Customer_Type=lambda df:
        df["Orders"].apply(
            lambda x: "Repeat" if x > 1 else "New"
        )
    )
)

customer_type = (
    customer_type
    .groupby("Customer_Type", as_index=False)
    .agg(
        Customers=("user_id", "count")
    )
)

customer_type.to_csv(
    "../data/processed/customers/customer_type.csv",
    index=False
)

customer_type

,Customer_Type,Customers
0,New,31105
1,Repeat,591


In [73]:
# 3.7.5 Customer Lifetime Value

customer_ltv = (
    orders
    .groupby("user_id", as_index=False)
    .agg(
        Total_Revenue=("price_usd", "sum"),
        Orders=("order_id", "count")
    )
)

customer_ltv["Average_Order_Value"] = (
    customer_ltv["Total_Revenue"] /
    customer_ltv["Orders"]
)

customer_ltv.to_csv(
    "../data/processed/customers/customer_ltv.csv",
    index=False
)

customer_ltv.head()

,user_id,Total_Revenue,Orders,Average_Order_Value
0,13,49.99,1,49.99
1,20,49.99,1,49.99
2,59,49.99,1,49.99
3,104,49.99,1,49.99
4,147,49.99,1,49.99


In [74]:
print(customer_summary.columns)

Index(['user_id', 'Revenue', 'Customer_Segment', 'Total_Orders',
       'Customer_Lifetime_Value'],
      dtype='object')


# 3.8 Business Insights

In [75]:
# ==========================================
# 3.8.1 Key Findings
# ==========================================

print("="*60)
print("KEY FINDINGS")
print("="*60)

print("""
1. The Original Mr. Fuzzy was the highest-performing product, generating approximately 62.47% of the total revenue.

2. The Birthday Sugar Panda recorded the highest refund rate (6.04%), indicating potential quality or customer satisfaction concerns.

3. The Hudson River Mini Bear had the lowest refund rate (1.28%), suggesting strong customer satisfaction despite lower sales.

4. The business served 31,696 customers with 32,313 total orders, averaging 1.02 orders per customer, indicating a predominantly one-time customer base.

5. Average revenue generated per customer was $61.16.

6. Revenue grew from 2012 to 2014. The 2015 results are year-to-date through March 19 and should not be compared with full-year totals.

7. Gsearch generated the highest website traffic (316,035 sessions) and the highest overall revenue ($1.28M), making it the most valuable acquisition channel.

8. Direct traffic achieved the highest conversion rate (7.34%), indicating stronger purchase intent compared to paid channels.

9. Socialbook generated the lowest conversion rate (3.21%), suggesting lower marketing efficiency.

10. Product pages received the highest pageviews, while the Thank You page naturally recorded a 100% exit rate as the final stage of the purchase journey.
""")

KEY FINDINGS

1. The Original Mr. Fuzzy was the highest-performing product, generating approximately 62.47% of the total revenue.

2. The Birthday Sugar Panda recorded the highest refund rate (6.04%), indicating potential quality or customer satisfaction concerns.

3. The Hudson River Mini Bear had the lowest refund rate (1.28%), suggesting strong customer satisfaction despite lower sales.

4. The business served 31,696 customers with 32,313 total orders, averaging 1.02 orders per customer, indicating a predominantly one-time customer base.

5. Average revenue generated per customer was $61.16.

6. Revenue grew from 2012 to 2014. The 2015 results are year-to-date through March 19 and should not be compared with full-year totals.

7. Gsearch generated the highest website traffic (316,035 sessions) and the highest overall revenue ($1.28M), making it the most valuable acquisition channel.

8. Direct traffic achieved the highest conversion rate (7.34%), indicating stronger purchase intent 

In [76]:
# ==========================================
# 3.8.2 Product Recommendations
# ==========================================

print("="*60)
print("PRODUCT RECOMMENDATIONS")
print("="*60)

print("""
• Continue promoting The Original Mr. Fuzzy as the flagship product while reducing dependency on a single product for revenue.

• Investigate the high refund rate of The Birthday Sugar Panda by reviewing product quality, packaging, or customer expectations.

• Increase visibility of The Hudson River Mini Bear through targeted promotions due to its low refund rate and positive customer satisfaction.

• Introduce product bundles to encourage sales of lower-performing products alongside best sellers.

• Continuously monitor product profitability and refund trends for inventory optimization.
""")

PRODUCT RECOMMENDATIONS

• Continue promoting The Original Mr. Fuzzy as the flagship product while reducing dependency on a single product for revenue.

• Investigate the high refund rate of The Birthday Sugar Panda by reviewing product quality, packaging, or customer expectations.

• Increase visibility of The Hudson River Mini Bear through targeted promotions due to its low refund rate and positive customer satisfaction.

• Introduce product bundles to encourage sales of lower-performing products alongside best sellers.

• Continuously monitor product profitability and refund trends for inventory optimization.



In [77]:
# ==========================================
# 3.8.3 Marketing Recommendations
# ==========================================

print("="*60)
print("MARKETING RECOMMENDATIONS")
print("="*60)

print("""
• Continue investing in Gsearch as it drives the highest traffic and revenue.

• Analyze why Direct traffic converts better and apply similar strategies to paid marketing campaigns.

• Improve Socialbook campaign performance by refining audience targeting and advertising creatives.

• Regularly monitor conversion rates across marketing channels to optimize advertising budgets.

• Perform A/B testing on marketing campaigns to improve customer acquisition efficiency.
""")

MARKETING RECOMMENDATIONS

• Continue investing in Gsearch as it drives the highest traffic and revenue.

• Analyze why Direct traffic converts better and apply similar strategies to paid marketing campaigns.

• Improve Socialbook campaign performance by refining audience targeting and advertising creatives.

• Regularly monitor conversion rates across marketing channels to optimize advertising budgets.

• Perform A/B testing on marketing campaigns to improve customer acquisition efficiency.



In [78]:
# ==========================================
# 3.8.4 Customer Recommendations
# ==========================================

print("="*60)
print("CUSTOMER RECOMMENDATIONS")
print("="*60)

print("""
• Implement a customer loyalty program to encourage repeat purchases.

• Launch personalized email campaigns targeting first-time customers to improve repeat purchase rates.

• Identify high-value customers and provide exclusive offers or early product access.

• Segment customers based on purchasing behavior for more personalized marketing.

• Monitor Customer Lifetime Value (CLV) regularly to guide customer retention strategies.
""")

CUSTOMER RECOMMENDATIONS

• Implement a customer loyalty program to encourage repeat purchases.

• Launch personalized email campaigns targeting first-time customers to improve repeat purchase rates.

• Identify high-value customers and provide exclusive offers or early product access.

• Segment customers based on purchasing behavior for more personalized marketing.

• Monitor Customer Lifetime Value (CLV) regularly to guide customer retention strategies.



In [79]:
# ==========================================
# 3.8.5 Website Recommendations
# ==========================================

print("="*60)
print("WEBSITE RECOMMENDATIONS")
print("="*60)

print("""
• Optimize high-traffic product pages to improve conversion rates.

• Review checkout and shipping pages to identify opportunities for reducing customer drop-offs.

• Continue testing landing page designs to improve visitor engagement.

• Improve product page content, images, and call-to-action elements.

• Regularly monitor page exits and user behavior using website analytics tools.
""")

WEBSITE RECOMMENDATIONS

• Optimize high-traffic product pages to improve conversion rates.

• Review checkout and shipping pages to identify opportunities for reducing customer drop-offs.

• Continue testing landing page designs to improve visitor engagement.

• Improve product page content, images, and call-to-action elements.

• Regularly monitor page exits and user behavior using website analytics tools.



In [80]:
# ==========================================
# 3.8.6 Future Improvements
# ==========================================

print("="*60)
print("FUTURE IMPROVEMENTS")
print("="*60)

print("""
• Develop sales forecasting models using historical sales data.

• Build a customer churn prediction model using Machine Learning.

• Implement personalized product recommendation systems.

• Create interactive dashboards using Power BI.

• Automate data pipelines for scheduled business reporting.

• Perform advanced customer segmentation using clustering techniques.

• Integrate real-time business monitoring dashboards for decision-making.
""")

FUTURE IMPROVEMENTS

• Develop sales forecasting models using historical sales data.

• Build a customer churn prediction model using Machine Learning.

• Implement personalized product recommendation systems.

• Create interactive dashboards using Power BI.

• Automate data pipelines for scheduled business reporting.

• Perform advanced customer segmentation using clustering techniques.

• Integrate real-time business monitoring dashboards for decision-making.

